In [1]:
# Read the fasta file
from Bio import SeqIO
import pandas as pd
file_path="../Python for Genomic/dna2.fasta"
records = []
dna_id = []
dna_seq = []
for record in SeqIO.parse(file_path,"fasta"):
    records.append({"id":record.id,
                    "sequence":str(record.seq),
                    "length":len(record.seq)
                   })
    dna_id.append(record.id)
    dna_seq.append(str(record.seq))

df = pd.DataFrame(records)

In [2]:
print(df.shape)
print(max(df["length"]))
print(min(df["length"]))

(18, 3)
4894
115


In [8]:
idx = df["length"].idxmax()
row = df.loc[df["length"].idxmax()]
print(row)
max_id = df.loc[df["length"].idxmax(),"id"]
print(max_id)

id                             gi|142022655|gb|EQ086233.1|255
sequence    CTCGACGCGCTCCGCGTCGAGGTCGCCCGACGTCTCGCGCAGCAAC...
length                                                   4894
Name: 2, dtype: object
gi|142022655|gb|EQ086233.1|255


In [ ]:
# Define all 6 reading frames
from Bio.Seq import Seq

def six_frames(dna):
    seq = Seq(dna)
    frames = [
        seq,
        seq[1:],
        seq[2:],
        seq.reverse_complement(),
        seq.reverse_complement()[1:],
        seq.reverse_complement()[2:]
    ]
    return frames
    

In [17]:
# Find ORFs in frame1 ,2 3
def find_orfs(seq):
    seq = str(seq)
    frames={}

    start_codon = "ATG"
    stop_codons = {"TAA","TAG","TGA"}
    

    for frame in range(3):          # check 3 frames
        i = frame
        orfs = []
        while i < len(seq)-2:

            codon = seq[i:i+3]

            if codon == start_codon:

                for j in range(i+3, len(seq)-2, 3):

                    codon2 = seq[j:j+3]

                    if codon2 in stop_codons:
                        orfs.append(seq[i:j+3])
                        break
        
            i += 3
        frames[frame] = orfs  

    return frames

    
df[["frame1","frame2","frame3"]] = df["sequence"].apply(find_orfs).apply(pd.Series)



In [24]:


# Find the longest ORF in Frame 2
def longest_orf_frame(orfs):
    the_longest_orf = ""
    for orf in orfs:
        if len(orf) > len(the_longest_orf):
            the_longest_orf = orf
    return the_longest_orf

df["longest_orf_frame2"] = df["frame2"].apply(longest_orf_frame) 

    
                    
    

In [26]:

df["orf_length_frame2"] = df["longest_orf_frame2"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)

In [27]:
print(max(df["orf_length_frame2"]))

1458


In [28]:
df["longest_orf_frame1"] = df["frame1"].apply(longest_orf_frame) 
df["orf_length_frame1"] = df["longest_orf_frame1"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)
print(max(df["orf_length_frame1"]))

2394


In [29]:
df["longest_orf_frame3"] = df["frame3"].apply(longest_orf_frame) 
df["orf_length_frame3"] = df["longest_orf_frame3"].apply(
    lambda x: len(x) if isinstance(x, str) else 0
)
print(max(df["orf_length_frame3"]))

1821


In [18]:
# Find the ORFs position in frame3
start_codon = "ATG"
stop_codons = {"TAA", "TAG", "TGA"}

def orfs_frame3(seq):
    orfs = []
    frame = 2 # frame 3 (0,1,2)

    for i in range(frame,len(seq)-2,3):
        codon = seq[i:i+3]

        if codon == start_codon:
            for j in range(i+3,len(seq)-2,3):
                stop = seq[j:j+3]

                if stop in stop_codons:
                    orfs.append({
                        "start": i,
                        "end": j+3,
                        "length" : j+3-i
                    })
                    break   
    

    return orfs


In [19]:
df["orfs_frame3"]= df["sequence"].apply(orfs_frame3)

In [20]:
orf_rows = []
for _, row in df.iterrows():
    for o in row["orfs_frame3"]:
        orf_rows.append({
            "seq_id": row["id"],
            "start": o["start"],
            "end": o["end"],
            "length": o["length"]
        })
orf_frame3_df = pd.DataFrame(orf_rows)
            
        

In [21]:
longest_orf = orf_frame3_df.loc[orf_frame3_df["length"].idxmax()]

In [23]:
start_position = longest_orf["start"]+1
print(start_position)

636


In [32]:
#What is the length of the longest forward ORF that appears in the sequence with the identifier  gi|142022655|gb|EQ086233.1|16?

new_df = df[df["id"]=="gi|142022655|gb|EQ086233.1|16"]

new_df

,id,sequence,length,frame1,frame2,frame3,longest_orf_frame2,orf_length_frame2,orfs_frame3,longest_orf_frame1,orf_length_frame1,longest_orf_frame3,orf_length_frame3
12,gi|142022655|gb|EQ086233.1|16,GTCGATCGACACGACGCTCGCGCAGCGCGACGCGAAGGCCGCGTGA...,4804,[ATGTCGTCAACGTCAGTTCGCGCTATGGCGCGGTGCAGTGGAACG...,[ATGGATCCTGCCGCTCGTCGAGGATACGGGCGCCGACGCGGTCGA...,[ATGGCCGACCTTCGCTGCACCATCGCGGGCATCACTTCGCCGAAC...,ATGGCAATCCTGATTCGTGGCGGCACCGTGGTCGATGCGGACCGTT...,1458,"[{'start': 110, 'end': 1427, 'length': 1317}, ...",ATGAATCACGCAGCGAATCCCGCCGATCCCGATCGCGCCGCGGCGC...,1509,ATGCGGGCCATCCTGCATCGCCGCCTTTCGTTCCACCCGGGCCGGC...,1644


In [41]:
#Find the most frequently occurring repeat of length 6 in all sequences. How many times does it occur in all?
from collections import Counter

def repeat_6mers(seq):
    k = 6
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    counts = Counter(kmers)

    # find the (kmer, count) pair with the highest count
    kmer, c = max(counts.items(), key=lambda x: x[1])
    return kmer, c


all_seq = "".join(df["sequence"])

max_count = repeat_6mers(all_seq)
max_count


('GCGCGC', 153)

In [46]:
from collections import Counter

def repeat_12mers(seq):
    k = 12
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    counts = Counter(kmers)

    max_count = max(counts.values())

    # return all kmers that have this max count
    max_kmers = [kmer for kmer, c in counts.items() if c == max_count]

    return max_kmers, max_count




repeat_12 = repeat_12mers(all_seq)
repeat_12

(['CATTCGCCATTC', 'ATTCGCCATTCG', 'TTCGCCATTCGC', 'TCGCCATTCGCC'], 10)

In [47]:
from collections import Counter

def repeat_7mers(seq):
    k = 7
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    counts = Counter(kmers)

    max_count = max(counts.values())

    # return all kmers that have this max count
    max_kmers = [kmer for kmer, c in counts.items() if c == max_count]

    return max_kmers, max_count




repeat_7 = repeat_7mers(all_seq)
repeat_7

(['CGCCGCG', 'CGCGCCG'], 63)